In [2]:
#MAximum Marginal relevance is an powerful diversity aware retriever  techniques used in information retriever and RAG pipelines to balance relevence and novelty when selecting documents

In [6]:
from langchain_community.vectorstores import FAISS

In [7]:
import os
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

In [9]:
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
#step1:Load and chunk the docuement in here
from langchain_community.document_loaders import TextLoader

loader=TextLoader('doc_0.txt')

In [15]:
loader

In [17]:
raw_docs=loader.load()
raw_docs

[Document(metadata={'source': 'doc_0.txt'}, page_content='\n    MachineLearning: "Machine learning is a field of artificial intelligence that focuses on creating algorithms and models that enable computers to learn patterns and relationships from data without being explicitly programmed. Instead of following fixed rules, machine learning systems improve their performance over time as they are exposed to more information. It encompasses techniques such as supervised learning, where models are trained on labeled data to make predictions, and unsupervised learning, where systems discover hidden structures in unlabeled data. Machine learning is widely used in applications like recommendation systems, fraud detection, predictive analytics, and medical diagnosis, making it a cornerstone of modern AI.",\n    ')]

In [19]:
#Splitter
from langchain_core.documents import Document
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

In [20]:
splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

In [23]:
chunk=splitter.split_documents(raw_docs)
chunk

[Document(metadata={'source': 'doc_0.txt'}, page_content='MachineLearning: "Machine learning is a field of artificial intelligence that focuses on creating algorithms and models that enable computers to learn patterns and relationships from data without'),
 Document(metadata={'source': 'doc_0.txt'}, page_content='from data without being explicitly programmed. Instead of following fixed rules, machine learning systems improve their performance over time as they are exposed to more information. It encompasses'),
 Document(metadata={'source': 'doc_0.txt'}, page_content='It encompasses techniques such as supervised learning, where models are trained on labeled data to make predictions, and unsupervised learning, where systems discover hidden structures in unlabeled'),
 Document(metadata={'source': 'doc_0.txt'}, page_content='in unlabeled data. Machine learning is widely used in applications like recommendation systems, fraud detection, predictive analytics, and medical diagnosis, making it

In [25]:
#using the vectors stores and the hugging face embeddings
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(
    model='sentence-transformers/all-MiniLM-L6-v2'
)

In [27]:
#Loading the vector stores
from langchain_community.vectorstores import FAISS
vectorstores=FAISS.from_documents(chunk,embedding=embeddings)

In [28]:
vectorstores

In [33]:
#Creating the retriever
retriever=vectorstores.as_retriever(
    search_type='mmr',
    search_kwargs={"k":3}
)

In [34]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002B862016C30>, search_type='mmr', search_kwargs={'k': 3})

In [ ]:
#Prompt template
from langchain_core.prompts import PromptTemplate
prompt=PromptTemplate.from_template(
"""Answer the following question based on the context that is provided.
Context: {context}
Question: {input}
"""
)

In [47]:
from langchain_groq import ChatGroq
llm=ChatGroq(model='groq:llama-3.1-8b-instant')

In [52]:
#Calling the chat model
from langchain.chat_models import init_chat_model
llm=init_chat_model(model="groq:llama-3.1-8b-instant")
llm 

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002B863EC5AF0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002B863EC4BC0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [53]:
llm.invoke("Hi")

AIMessage(content="It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 36, 'total_tokens': 59, 'completion_time': 0.027098036, 'completion_tokens_details': None, 'prompt_time': 0.001592728, 'prompt_tokens_details': None, 'queue_time': 0.050887442, 'total_time': 0.028690764}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b832c-42a1-7601-9b70-4c49652ded13-0', usage_metadata={'input_tokens': 36, 'output_tokens': 23, 'total_tokens': 59})

In [56]:
#Making an RAG pipeline
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Create document chain
document_chain = create_stuff_documents_chain(llm,prompt=prompt)

In [57]:
#This is creating an create_retrieve_chain 
rag_chain=create_retrieval_chain(retriever=retriever,combine_docs_chain=document_chain)

In [1]:
#Step-6-QUERY
query={"input":"How does Langchain supports agents and the memeory?"}
response=rag_chain.invoke(query)

NameError: name 'rag_chain' is not defined